# 🔧 Ratchet Fix Model — Fine-Tuning
Fine-tunes Qwen 3.5 4B on 1,644 code fix examples using LoRA.
Exports to GGUF Q4_K_M for local Ollama deployment.

**Runtime:** T4 GPU (Colab free tier) · ~20 min training · ~5 min export

**Steps:**
1. Upload `ratchet-fix-augmented.jsonl` when prompted
2. Click Runtime → Run all
3. Download the GGUF file at the end

In [ ]:
# Step 1: Install dependencies
!pip install -q unsloth[colab] huggingface_hub

In [ ]:
# Step 2: Upload training data
from google.colab import files
import json

print("Upload ratchet-fix-augmented.jsonl...")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

with open(filename) as f:
    data = [json.loads(l) for l in f]
print(f"Loaded {len(data)} examples")

# Verify format
print(f"Keys: {data[0].keys()}")
print(f"First message roles: {[m['role'] for m in data[0]['messages']]}")

In [ ]:
# Step 3: Split data
import random
random.seed(42)
random.shuffle(data)

n = len(data)
train_end = int(n * 0.90)
val_end = int(n * 0.95)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

In [ ]:
# Step 4: Load model with Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(model.print_trainable_parameters())

In [ ]:
# Step 5: Prepare dataset
from datasets import Dataset

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

train_dataset = Dataset.from_list(train_data).map(format_chat)
val_dataset = Dataset.from_list(val_data).map(format_chat)

print(f"Sample (first 500 chars):\n{train_dataset[0]['text'][:500]}")

In [ ]:
# Step 6: Train
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=TrainingArguments(
        output_dir="./ratchet-fix-output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=50,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=200,
        save_total_limit=2,
        fp16=True,
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
    ),
    max_seq_length=2048,
    dataset_text_field="text",
    packing=True,
)

print("Starting training...")
stats = trainer.train()
print(f"\nDone! Train loss: {stats.training_loss:.4f}")

In [ ]:
# Step 7: Quick test before export
FastLanguageModel.for_inference(model)

test_prompt = """You are a code quality assistant. Given a code snippet with an issue, provide the fixed version.

Fix the empty catch block in this code:

```typescript
try {
  await db.query('SELECT * FROM users');
} catch (e) {
}
```"""

messages = [{"role": "user", "content": test_prompt}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

output = model.generate(inputs, max_new_tokens=512, temperature=0.7)
response = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)
print("Model response:")
print(response)

In [ ]:
# Step 8: Export to GGUF Q4_K_M
print("Exporting to GGUF Q4_K_M (this takes ~5 min)...")
model.save_pretrained_gguf(
    "ratchet-fix-4b",
    tokenizer,
    quantization_method="q4_k_m",
)
print("Export complete!")

import os
gguf_files = [f for f in os.listdir("ratchet-fix-4b") if f.endswith(".gguf")]
for f in gguf_files:
    size = os.path.getsize(f"ratchet-fix-4b/{f}") / 1e9
    print(f"{f}: {size:.2f} GB")

In [ ]:
# Step 9: Download the GGUF file
from google.colab import files

gguf_path = f"ratchet-fix-4b/{gguf_files[0]}"
print(f"Downloading {gguf_path}...")
files.download(gguf_path)
print("\nDone! Next steps:")
print("1. Create a Modelfile:")
print('   echo "FROM ./ratchet-fix-4b.gguf" > Modelfile')
print("2. Import to Ollama:")
print("   ollama create ratchet-fix:4b -f Modelfile")
print("3. Test:")
print('   ollama run ratchet-fix:4b "Fix this empty catch block..."')